In [ ]:
# In this file, we do SFT on Llama32-3B-Instruct model

In [ ]:
# download LLama3.2-3B-Instruct model

from huggingface_hub import snapshot_download

local_dir = snapshot_download(
    repo_id="meta-llama/Llama-3.2-3B-Instruct",
    local_dir="Llama3.2-3B-Instruct",
    local_dir_use_symlinks=False,
)
print(f"Model downloaded to {local_dir}")

In [ ]:
# SFT on LLama3.2-3B-Instruct model

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig


base_model_path = "Llama3.2-3B-Instruct"   
data_path = {
    "train": "validation_QTSA_mix/QTSA_train_sft.jsonl",
    "validation": "validation_QTSA_mix/QTSA_val_sft.jsonl"
}
output_dir = "outputs/(new)Llama3.2-3B-Instruct-qtsa-mix"

tokenizer = AutoTokenizer.from_pretrained(base_model_path, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

max_memory = {
    0: "30GiB",
    1: "30GiB",
}

# ==== load base model ====
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    device_map="auto",
    max_memory=max_memory,
    torch_dtype=torch.float16,
    use_cache=False,
)

base_model.gradient_checkpointing_enable()

# ==== LoRA config ====
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

# ==== load dataset ====
dataset = load_dataset("json", data_files=data_path)
train_dataset = dataset["train"]
eval_dataset = dataset["validation"]

# ==== SFT config ====
sft_config = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-5, 
    warmup_ratio=0.1,
    fp16=True,   # fp16
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    eval_strategy="steps",   
    eval_steps=50,
    lr_scheduler_type="cosine",
    load_best_model_at_end=True,   
    metric_for_best_model="eval_loss",  
    greater_is_better=False,       
    logging_dir="logs/(new)Llama3.2-3B-Instruct-qtsa-mix",
    report_to="tensorboard",
    max_length=3000,
)

def formatting_func(ex):
    return (
        "<|begin_of_text|>"
        "<|start_header_id|>user<|end_header_id|>\n"
        f"{ex['instruction']}\n"
        "<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n"
        f"{ex['response']}\n"
        "<|eot_id|>"
    )

# ==== Trainer ====
trainer = SFTTrainer(
    model=base_model,
    formatting_func=formatting_func,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,     
    processing_class=tokenizer,
    peft_config=peft_config,
)

trainer.train()

# ==== save model ====
model.save_pretrained(f"{output_dir}")
tokenizer.save_pretrained(f"{output_dir}")

In [ ]:
trainer.model.save_pretrained(f"{output_dir}")
tokenizer.save_pretrained(f"{output_dir}")